# 1IEP rooted representation controls

Owning DockingMT issue #17. Inspect six preserved native searches and twelve new searches, plus a separate raw-input repeat. Physical input geometry, chemistry and seven cuts agree. Reversing ROOT changes its first atom and therefore the coordinate origin in the official Vina parser. No preferred ROOT, convergence or affinity claim.

In [1]:
from pathlib import Path
import gzip, hashlib, json
import numpy as np
import pyunitwizard as puw
import dockingmt as dmt
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists())
path = root / 'devguide/validation/data/1iep_representation/audit_2026-10-07.json.gz'
raw = gzip.decompress(path.read_bytes())
record = json.loads(raw)
assert len(record['original_native_runs']) == 6 and len(record['new_runs']) == 12
assert hashlib.sha256((root / record['baseline']['path']).read_bytes()).hexdigest() == record['baseline']['sha256']
print('Retained producer:', record['baseline']['source_producer'])
print('Original JSON SHA-256:', hashlib.sha256(raw).hexdigest())
for variant, pose in record['fixed_conformation_scores'].items():
    print(variant, pose['scores'])
    assert dmt.verify_captured_inputs(pose['metadata']['scoring_history'][0]['backend_artifacts'])

Retained producer: 7b11391e153f77f340fd039bc799352f90da83d3
Original JSON SHA-256: 1dccf14aa550a58a4d0a82b8ddf2b0fd77f9c206524bc99c1151e0eeab6d5113
native {'vina': -12.513, 'vina.flex_inter': 0.0, 'vina.flex_intra': 0.0, 'vina.lig_inter': -17.634, 'vina.lig_intra': -0.485, 'vina.lig_intra_best_pose': -0.485, 'vina.other_inter': 0.0, 'vina.torsions': 5.121}
native_root_reversed {'vina': -12.513, 'vina.flex_inter': 0.0, 'vina.flex_intra': 0.0, 'vina.lig_inter': -17.634, 'vina.lig_intra': -0.485, 'vina.lig_intra_best_pose': -0.485, 'vina.other_inter': 0.0, 'vina.torsions': 5.121}
published {'vina': -12.513, 'vina.flex_inter': 0.0, 'vina.flex_intra': 0.0, 'vina.lig_inter': -17.634, 'vina.lig_intra': -0.485, 'vina.lig_intra_best_pose': -0.485, 'vina.other_inter': 0.0, 'vina.torsions': 5.121}


In [2]:
print('variant               effort seed poses first-heavy closest-heavy first-score')
for run in record['original_native_runs'] + record['new_runs']:
    name = 'native' if run['variant'] == 'flexible' else run['variant']
    p = run['result']['protocol_info']['parameters']
    heavy = run['heavy_atom_metrics']
    values = [row['heavy_atom_positional_rmsd_angstrom'] for row in heavy]
    print(f"{name:21s} {p['exhaustiveness']:6d} {p['seed']:4d} {len(heavy):5d} {values[0]:11.4f} {min(values):13.4f} {heavy[0]['vina_score_kcal_per_mol']:11.3f}")
    assert dmt.verify_captured_inputs(run['result']['provenance']['backend_artifacts'])
print('RMSD in angstrom, no alignment or symmetry correction; scores are Vina kcal/mol.')

variant               effort seed poses first-heavy closest-heavy first-score
native                     1    7     3     12.3453       12.3453     -10.899
native                     1   42     1     12.3663       12.3663     -10.779
native                     1 2026     3     12.3183       12.1256     -10.888
native                     8    7     3      0.2928        0.2928     -13.302
native                     8   42     5     12.3349       12.0659     -10.827
native                     8 2026     5     12.3183       12.0296     -10.888
published                  1    7     1      0.2718        0.2718     -13.293
published                  1   42     1      0.8969        0.8969     -13.286
published                  1 2026     3     12.3188       12.1504     -10.945
published                  8    7     4      0.9045        0.9045     -13.227
published                  8   42     4      0.8969        0.8969     -13.286
published                  8 2026     2      0.9045        0.904

In [3]:
run = next(r for r in record['new_runs'] if r['variant'] == 'published')
restored = dmt.DockingResult.from_dict(run['result'])
xyz = puw.get_value(puw.quantity(record['audit']['source_snapshot']['structures']['coordinates'], 'nm'), to_unit='angstrom')[0]
reference = puw.quantity(xyz[run['pdbqt_to_source_atom_indices']], 'angstrom')
with puw.context(standard_units=['pm', 'fs', 'K', 'mole', 'dalton', 'e', 'kJ/mol', 'radians']):
    report = dmt.evaluate_redocking(restored, reference, rmsd_cutoff=puw.quantity(250, 'pm'))
assert report['criterion'] == run['evaluation']['criterion']
np.testing.assert_allclose([r['rmsd'] for r in report['poses']], [r['rmsd'] for r in run['evaluation']['poses']], rtol=0, atol=1e-12)
print('Saved written-order report agrees under pm/fs; no engine searches rerun.')

Saved written-order report agrees under pm/fs; no engine searches rerun.
